Paper link: https://arxiv.org/pdf/2102.06171

Key Terms:

Problem: Batch norm has many problems:
- computationally expensive
- introduces discrepancy between training and inference
- batch size dependent

Remind me what Batch Norm is: downscales the residual branch
- reduces the F(x) at initialization to be close to identity mapping, so that training becomes stable.

- reduces mean shift, which is the change in the distribution of layer inputs during training, as the parameters of the previous layers change. We don't want the deep networks to predict the same label for all inputs.

- regularizer - helps with optimization

Solution Architecture:

AGC applied to every layer except the last linear layer.

Mixup, Cutmix, RandAugment used for data augmentation.

Scaled Weight Standardization (Scaled W) applied to all convolutional layers.

- Scaled Weight Standardization normalizes the weights of convolutional layers by their standard deviation and scales them by a learnable parameter. This helps stabilize training by ensuring that the weights have a consistent scale, which can improve convergence and performance.

This method makes sure that each layer never produce unstable outputs.

- Think of each convolution filter as a “loudspeaker.”
- Without control: some layers shout, some whisper
- BatchNorm listens and adjusts volume dynamically
- Scaled WS builds every speaker to output at the same volume



In [1]:
import torch
from torch import nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
import matplotlib.pyplot as plt


class ScaledStdConv2d(nn.Conv2d):
    """Scaled Weight Standardized Conv2d with fixed beta scaling."""
    def __init__(self, *args, bias=False, **kwargs):
        super().__init__(*args, bias=bias, **kwargs)
    
    def forward(self, x):
        weight = self.weight
        mean = weight.mean(dim=[1, 2, 3], keepdim=True)
        std = weight.std(dim=[1, 2, 3], keepdim=True) + 1e-5
        weight = (weight - mean) / std
        # Fixed beta for tau=1.0, 2 nonlinearities, 3 convs: (1 / 0.25)^{1/6} ≈ 1.26
        beta = 1.26
        weight = weight * beta
        return F.conv2d(x, weight, None, self.stride, self.padding, self.dilation, self.groups)

class SqueezeExcite(nn.Module):
    def __init__(self, channels, se_ratio=0.5):
        super().__init__()
        se_channels = max(1, int(channels * se_ratio))
        self.fc1 = nn.Linear(channels, se_channels)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(se_channels, channels)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        y = F.adaptive_avg_pool2d(x, (1, 1)).squeeze(-1).squeeze(-1)
        y = self.fc1(y)
        y = self.relu(y)
        y = self.fc2(y)
        y = self.sigmoid(y)
        y = y.unsqueeze(-1).unsqueeze(-1)
        return x * y

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, expansion=2, group_width=128, se_ratio=0.5, alpha=0.2, dropout_rate=0.2):
        super().__init__()
        mid_channels = out_channels * expansion
        groups = mid_channels // group_width if group_width > 0 else 1
        
        self.alpha = alpha
        self.gamma1 = nn.Parameter(torch.tensor(1.0))
        self.gamma2 = nn.Parameter(torch.tensor(1.0))
        self.gamma3 = nn.Parameter(torch.tensor(1.0))
        
        self.conv1 = ScaledStdConv2d(in_channels, mid_channels, kernel_size=1, stride=1, padding=0)
        self.act1 = nn.GELU()
        
        self.conv2 = ScaledStdConv2d(mid_channels, mid_channels, kernel_size=3, stride=stride, padding=1, groups=groups)
        self.act2 = nn.GELU()
        
        self.conv3 = ScaledStdConv2d(mid_channels, out_channels, kernel_size=1, stride=1, padding=0)
        
        self.se = SqueezeExcite(out_channels, se_ratio=se_ratio)
        self.dropout = nn.Dropout(dropout_rate)
        
        # Fix: Add padding=1 to AvgPool2d to ensure dimensions match
        if stride != 1 or in_channels != out_channels:
            self.skip = ScaledStdConv2d(
                in_channels, out_channels,
                kernel_size=1, stride=stride, padding=0, bias=False
            )
        else:
            self.skip = nn.Identity()
    
    def forward(self, x):
        residual = self.skip(x)
        
        y = self.conv1(x) * self.gamma1
        y = self.act1(y)
        
        y = self.conv2(y) * self.gamma2
        y = self.act2(y)
        
        y = self.conv3(y) * self.gamma3
        
        y = self.se(y)
        y = self.dropout(y)
        
        y = y * self.alpha
        
        return y + residual

class NFNet(nn.Module):
    def __init__(self, num_classes=100, variant='F0'):
        super().__init__()
        if variant == 'F0':
            widths = [256, 512, 1024, 1536]
            depths = [1, 2, 6, 3]
            group_width = 128
            alpha = 0.2
            se_ratio = 0.5
            expansion = 2
            dropout_rate = 0.2
        
        # Stem with gammas
        self.stem_gamma1 = nn.Parameter(torch.tensor(1.0))
        self.stem_gamma2 = nn.Parameter(torch.tensor(1.0))
        self.stem_gamma3 = nn.Parameter(torch.tensor(1.0))
        self.stem_gamma4 = nn.Parameter(torch.tensor(1.0))
        
        self.stem_conv1 = ScaledStdConv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.stem_act1 = nn.GELU()
        
        self.stem_conv2 = ScaledStdConv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.stem_act2 = nn.GELU()
        
        self.stem_conv3 = ScaledStdConv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.stem_act3 = nn.GELU()
        
        self.stem_conv4 = ScaledStdConv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.stem_act4 = nn.GELU()
        
        # Body
        stages = []
        current_channels = 128   # after stem
        for stage_idx, (width, depth) in enumerate(zip(widths, depths)):
            for block_idx in range(depth):
                stride = 2 if block_idx == 0 and stage_idx >= 1 else 1   # ← delay first downsample, or even remove some
                # or more aggressively: stride = 2 if stage_idx == 2 else 1  (only downsample twice total)
                block = ResidualBlock(
                    current_channels, width, stride=stride,
                    # ... rest same
                )
                stages.append(block)
                current_channels = width

        self.body = nn.Sequential(*stages)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(current_channels, num_classes)   # 1536 for F0-like
        # He init for fc
        nn.init.kaiming_normal_(self.fc.weight, mode='fan_out', nonlinearity='relu')
    
    def forward(self, x):
        # Stem
        x = self.stem_conv1(x) * self.stem_gamma1
        x = self.stem_act1(x)
        
        x = self.stem_conv2(x) * self.stem_gamma2
        x = self.stem_act2(x)
        
        x = self.stem_conv3(x) * self.stem_gamma3
        x = self.stem_act3(x)
        
        x = self.stem_conv4(x) * self.stem_gamma4
        x = self.stem_act4(x)
        
        x = self.body(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [ ]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Mild to avoid over-distortion
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4865, 0.4409], std=[0.2673, 0.2564, 0.2761]),
    transforms.RandomErasing(p=0.25)  # Apply after normalization for consistency
])

test_transform = transforms.Compose([
    transforms.ToTensor(), # Moved ToTensor before Normalize (good practice)
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

# Load raw datasets
cifar_train_raw = datasets.CIFAR100(root="./data", train=True, download=True, transform=None)

train_size = int(0.9 * len(cifar_train_raw))  # 48,000

train_indices = list(range(0, train_size))
val_indices = list(range(train_size, len(cifar_train_raw)))

# Create datasets with appropriate transforms
cifar_train = Subset(
    datasets.CIFAR100(root="./data", train=True, transform=train_transform),
    train_indices
)
cifar_val = Subset(
    datasets.CIFAR100(root="./data", train=True, transform=test_transform),
    val_indices
)
# Use original test set (10,000 samples) - close to 10% of 60,000
cifar_test = datasets.CIFAR100(root="./data", train=False, transform=test_transform)

train_loader = DataLoader(
    cifar_train,
    batch_size=16,  # Changed from 512
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=6
)

val_loader = DataLoader(
    cifar_val,  # Use directly
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=6
)

test_loader = DataLoader(
    cifar_test,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=6
)

num_classes = 100

model = NFNet().to(device)

num_epochs = 40
loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)

'''
Our training used asynchronous stochastic gradient descent with 0.9 momentum [17], 
fixed learning rate schedule (decreasing the learning rate by 4% every 8 epochs). 
'''

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,  # Changed from 1e-3
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-2,  # Changed from 3e-3
    epochs=num_epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25.0,
    final_div_factor=1000.0
)

train_losses = []
val_losses = []
epochs_recorded = []

best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f'Starting Epoch {epoch+1}')
    model.train()

    current_loss = 0.0
    num_batches = 0

    for i, data in enumerate(train_loader):
        inputs, targets = data
        inputs, targets = inputs.to(device), targets.to(device)
            
        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        loss.backward()

        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()  # Step per batch for OneCycleLR

        current_loss += loss.item()
        num_batches += 1

        if i % 50 == 0:
            torch.cuda.empty_cache()
            print(f'Batch {i}/{len(train_loader)}, Loss: {loss.item():.4f}')

    avg_train_loss = current_loss / num_batches

    # Validate EVERY epoch (removed the if condition)
    model.eval()
    val_loss = 0.0
    val_batches = 0

    with torch.no_grad():
        for val_data in val_loader:
            val_inputs, val_targets = val_data
            val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)

            val_outputs = model(val_inputs)
            val_batch_loss = loss_function(val_outputs, val_targets)

            val_loss += val_batch_loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches
    best_val_loss = min(best_val_loss, avg_val_loss)
    
    # Record metrics every epoch
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    epochs_recorded.append(epoch + 1)

    print(f'Epoch {epoch+1} - Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}')

if torch.cuda.is_available():
    torch.cuda.empty_cache()



Using device: cuda
GPU Memory: 8.2 GB


/home/colin-zhou/miniforge3/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Starting Epoch 1


OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 313.19 MiB is free. Including non-PyTorch memory, this process has 7.30 GiB memory in use. Of the allocated memory 7.03 GiB is allocated by PyTorch, and 155.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def evaluate_test_set(model):
    model.eval()
    correct = 0
    total = 0
    
    print("Starting evaluation...")
    
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            
            # Use autocast for consistency if you trained with it
            outputs = model(images)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the network on the test images: {accuracy:.2f}%')
    return accuracy

print("\n=== Running standard evaluation ===")
standard_accuracy = evaluate_test_set(model)   
print(f'Standard Test Accuracy: {standard_accuracy:.4f} ({standard_accuracy*100:.2f}%)')